# Extraction and alignment

Event-window extraction (8x8 windows, +-1 h of peaks), sharded downloads, timing-lag checks, storm-type labelling.

Cells are the original session code in execution order (exported verbatim; only secrets handling and artifact paths were changed). Each cell starts with a comment giving its original index, conda environment and run status. Exploratory and failed cells are kept, so later cells that rely on their partial state still make sense.


In [ ]:
# Setup: run from the repository root; helper modules live in prod/
import os, sys
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
sys.path.insert(0, 'prod')
os.makedirs('out', exist_ok=True)
from style import apply_figure_style  # stand-in for the figure style used in the session


In [65]:
# [orig cell 64 | env: precip | status: ok]
import json, sys
sys.path.insert(0,'prod'); import importlib, common; importlib.reload(common)
Hneed=pd.read_parquet('out/needed_hours.parquet').hour_start_utc
hpos=pd.Series(np.arange(len(Hneed)),index=Hneed)
Vst=pd.read_csv('out/valid_stations.csv'); sidx={s:i for i,s in enumerate(Vst.id)}
Hv=pd.read_parquet('ghcnh/fl_hourly_valid.parquet')
G=np.full((len(Hneed),len(Vst)),np.nan,np.float32)   # gauge matrix on needed hours
m=Hv[Hv.hr.isin(hpos.index)]; G[hpos[m.hr].values, m.STATION.map(sidx).values]=m.precipitation.values
print('gauge coverage on window hours', np.isfinite(G).mean().round(3))
def load(name):
    A=np.load(f'prod/{name}_win.npy'); meta=json.load(open(f'prod/{name}_meta.json'))
    assert meta['stations']==list(Vst.id)
    masks=[common.neighborhood_masks(w) for w in meta['widx']]
    NB={}
    for N in ['N1','N4','N9','N16']:
        M=np.stack([masks[k][N] for k in range(len(masks))])   # (ns,8,8)
        Am=np.where(M[None],A,np.nan)
        with np.errstate(all='ignore'):
            NB[N+'max']=np.nanmax(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)
            NB[N+'mean']=np.nanmean(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)
    return NB, meta
# consecutive-hour lookup for lags
def shift(X,lag):
    """value at hour h+lag aligned to h (NaN if h+lag not in needed set)"""
    tgt=Hneed+pd.Timedelta(hours=lag); p=hpos.reindex(tgt).values
    out=np.full_like(X,np.nan); ok=~np.isnan(p); out[ok]=X[p[ok].astype(int)]; return out
def lagcorr(NB):
    r={}
    for lag in range(-2,3):
        Y=shift(NB['N1max'],lag); ok=np.isfinite(Y)&np.isfinite(G)
        r[lag]=np.corrcoef(G[ok],Y[ok])[0,1].round(3)
    return r
NBm,mm_=load('merra2'); print('merra2 lag corr', lagcorr(NBm))

gauge coverage on window hours 0.894
merra2 lag corr {-2: np.float64(0.198), -1: np.float64(0.214), 0: np.float64(0.226), 1: np.float64(0.229), 2: np.float64(0.222)}


In [66]:
# [orig cell 65 | env: precip | status: ok]
EV={T:pd.read_csv(f'out/events_ge{T}.csv',parse_dates=['peak_hour_utc']) for T in [20,30,40,50]}
for T in EV:
    e=EV[T]; e['flag']=e.flag.fillna(''); e['k']=hpos.reindex(e.peak_hour_utc).values; e['s']=e.STATION.map(sidx)
def score(name, NB, boot=200, rng=np.random.default_rng(0)):
    rows=[]; per_event=[]
    sh={(N,lag):shift(NB[N+'max'],lag) for N in ['N1','N4','N9','N16'] for lag in (-1,0,1)}
    shm={N:NB[N+'mean'] for N in ['N1','N4','N9','N16']}
    for T,e in EV.items():
        e=e[(e.flag=='')&e.k.notna()]; k=e.k.astype(int).values; s=e.s.values
        for N in ['N1','N4','N9','N16']:
            v0=sh[(N,0)][k,s]; vt=np.nanmax(np.stack([sh[(N,l)][k,s] for l in (-1,0,1)]),0)
            vmean=shm[N][k,s]
            ok=np.isfinite(v0)
            g=e.peak_mm_h.values
            d=dict(product=name,threshold=T,nbhd=N,n_events=int(ok.sum()),
                   hit_strict_0h=np.mean(v0[ok]>=T), hit_strict_1h=np.mean(vt[ok]>=T),
                   hit_ge10_0h=np.mean(v0[ok]>=10), hit_ge1_0h=np.mean(v0[ok]>=1),
                   ratio_median=np.median(v0[ok]/g[ok]), ratio_q25=np.quantile(v0[ok]/g[ok],.25), ratio_q75=np.quantile(v0[ok]/g[ok],.75),
                   ratio_mean_median=np.nanmedian(vmean[ok]/g[ok]))
            # station-block bootstrap CI on strict hit (0h)
            st=s[ok]; hit=(v0[ok]>=T).astype(float); us=np.unique(st); grp={u:np.where(st==u)[0] for u in us}
            bs=[]
            for b in range(boot):
                pick=rng.choice(us,len(us)); idx=np.concatenate([grp[u] for u in pick]); bs.append(hit[idx].mean())
            d['hit_strict_0h_lo'],d['hit_strict_0h_hi']=np.quantile(bs,[.025,.975])
            # contingency on all window hours (station-hours with gauge data)
            Y=sh[(N,0)]; okc=np.isfinite(Y)&np.isfinite(G)
            o=G[okc]>=T; f=Y[okc]>=T; a=(o&f).sum(); b_=(f&~o).sum(); c=(o&~f).sum()
            d.update(POD_hours=a/(a+c) if a+c else np.nan, FAR_hours=b_/(a+b_) if a+b_ else np.nan, CSI_hours=a/(a+b_+c) if a+b_+c else np.nan,
                     freq_bias_hours=(a+b_)/(a+c) if a+c else np.nan)
            rows.append(d)
            pe=e[ok][['event_id','STATION','peak_hour_utc','peak_mm_h']].copy(); pe['product']=name; pe['threshold']=T; pe['nbhd']=N
            pe['prod_max_0h']=v0[ok]; pe['prod_max_pm1h']=vt[ok]; pe['prod_mean_0h']=vmean[ok]; per_event.append(pe)
    return pd.DataFrame(rows), pd.concat(per_event)
sc_m,pe_m=score('MERRA-2',NBm,boot=50)
print(sc_m[['threshold','nbhd','n_events','hit_strict_0h','hit_strict_1h','hit_ge10_0h','hit_ge1_0h','ratio_median','POD_hours','FAR_hours','CSI_hours']].round(3).to_string(index=False))

 threshold nbhd  n_events  hit_strict_0h  hit_strict_1h  hit_ge10_0h  hit_ge1_0h  ratio_median  POD_hours  FAR_hours  CSI_hours
        20   N1      7407          0.001          0.002        0.012       0.323         0.021      0.002      0.890      0.002
        20   N4      7407          0.002          0.005        0.023       0.425         0.028      0.005      0.898      0.005
        20   N9      7407          0.005          0.009        0.034       0.480         0.033      0.009      0.917      0.008
        20  N16      7407          0.007          0.013        0.041       0.534         0.038      0.011      0.938      0.010
        30   N1      2944          0.000          0.000        0.017       0.343         0.017      0.001      0.900      0.001
        30   N4      2944          0.000          0.001        0.030       0.437         0.022      0.002      0.928      0.001
        30   N9      2944          0.001          0.003        0.042       0.494         0.025      0.00

In [67]:
# [orig cell 66 | env: precip | status: ok]
w=mm_['widx'][sidx['USW00012839']]; print('Miami win centre', np.array(w['win_lat'])[3,3], np.array(w['win_lon'])[3,3], w['dist_km'])
A=np.load('prod/merra2_win.npy')
for hh in [20,21,22,23]:
    k=hpos.get(pd.Timestamp(f'2020-05-26 {hh}:00')); print(hh, G[k,sidx['USW00012839']], A[k,sidx['USW00012839'],2:6,2:6].round(1).tolist())
# compare directly with raw opendap test file
m2=xr.open_dataset('m2_test.nc4'); print((m2.PRECTOTCORR.sel(lat=25.8,lon=-80.3,method='nearest')*3600).values[20:24].round(2))

Miami win centre 26.0 -80.625 38.79466112694797
20 3.3 [[0.5, 0.5, 0.4000000059604645, 0.20000000298023224], [0.30000001192092896, 0.6000000238418579, 0.4000000059604645, 0.20000000298023224], [0.20000000298023224, 0.4000000059604645, 0.5, 0.30000001192092896], [0.20000000298023224, 0.4000000059604645, 0.6000000238418579, 0.4000000059604645]]
21 78.5 [[0.6000000238418579, 0.6000000238418579, 0.4000000059604645, 0.20000000298023224], [0.30000001192092896, 0.6000000238418579, 0.4000000059604645, 0.20000000298023224], [0.20000000298023224, 0.4000000059604645, 0.4000000059604645, 0.20000000298023224], [0.20000000298023224, 0.4000000059604645, 0.5, 0.30000001192092896]]
22 84.6 [[0.699999988079071, 0.800000011920929, 0.6000000238418579, 0.30000001192092896], [0.30000001192092896, 0.699999988079071, 0.4000000059604645, 0.20000000298023224], [0.20000000298023224, 0.4000000059604645, 0.30000001192092896, 0.20000000298023224], [0.20000000298023224, 0.30000001192092896, 0.5, 0.30000001192092896]

In [68]:
%%bash
# [orig cell 67 | env: precip | status: ok]
python prod/nasa_od.py nldas2 2>&1 | tail -15

stations ok 75
2000 2.7 min
4000 5.9 min
6000 8.5 min
8000 11.1 min
10000 13.7 min
12000 16.3 min
14000 21.2 min
16000 28.3 min
18000 32.0 min
20000 35.6 min
22000 38.7 min
nldas2 done; fails 0 hours with data 22744 of 22744 40.4 min


In [69]:
# [orig cell 68 | env: precip | status: ok]
NBn,mn_=load('nldas2'); print('nldas lag', lagcorr(NBn))
print([w['id'] for w in mn_['widx'] if not w['ok']])

nldas lag {-2: np.float64(0.099), -1: np.float64(0.163), 0: np.float64(0.386), 1: np.float64(0.664), 2: np.float64(0.271)}
['USC00088841', 'USW00012836', 'USW00012850', 'USW00012896']


<kernel:40>:19: RuntimeWarning: All-NaN slice encountered
  NB[N+'max']=np.nanmax(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)
<kernel:40>:20: RuntimeWarning: Mean of empty slice
  NB[N+'mean']=np.nanmean(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)

In [70]:
# [orig cell 69 | env: precip | status: ok]
grp=Vst.source_group.values
def lagcorr_g(NB):
    out={}
    for gname in ['metar','onhour']:
        cs=grp==gname; r={}
        for lag in range(-2,3):
            Y=shift(NB['N1max'],lag)[:,cs]; X=G[:,cs]; ok=np.isfinite(Y)&np.isfinite(X); r[lag]=round(np.corrcoef(X[ok],Y[ok])[0,1],3)
        out[gname]=r
    return out
print('nldas',lagcorr_g(NBn)); print('merra',lagcorr_g(NBm))

nldas {'metar': {-2: np.float64(0.091), -1: np.float64(0.151), 0: np.float64(0.362), 1: np.float64(0.726), 2: np.float64(0.297)}, 'onhour': {-2: np.float64(0.112), -1: np.float64(0.182), 0: np.float64(0.426), 1: np.float64(0.559), 2: np.float64(0.228)}}
merra {'metar': {-2: np.float64(0.192), -1: np.float64(0.209), 0: np.float64(0.223), 1: np.float64(0.229), 2: np.float64(0.224)}, 'onhour': {-2: np.float64(0.208), -1: np.float64(0.223), 0: np.float64(0.232), 1: np.float64(0.23), 2: np.float64(0.218)}}


In [71]:
# [orig cell 70 | env: precip | status: ok]
Ys={lag:shift(NBn['N1max'],lag) for lag in range(-3,4)}
res=[]
for j,sid in enumerate(Vst.id):
    rr={}
    for lag,Y in Ys.items():
        ok=np.isfinite(Y[:,j])&np.isfinite(G[:,j])
        rr[lag]=np.corrcoef(G[ok,j],Y[ok,j])[0,1] if ok.sum()>500 else np.nan
    s=pd.Series(rr); res.append(dict(id=sid,grp=grp[j],src=Hv[Hv.STATION==sid].grp.iloc[0] if False else '',best=s.idxmax() if s.notna().any() else np.nan,rbest=s.max(),r0=s.get(0),r1=s.get(1),r2=s.get(2)))
LG=pd.DataFrame(res); print(pd.crosstab(LG.grp,LG.best))
print(LG[(LG.grp=='onhour')&(LG.best!=1)].round(2).to_string())

best    -3.0   0.0   1.0
grp                     
metar      1     2    42
onhour     0     4    26
             id     grp src  best  rbest    r0    r1    r2
3   USC00082229  onhour       0.0   0.44  0.44  0.42  0.22
4   USC00084095  onhour       0.0   0.54  0.54  0.51  0.20
6   USC00084802  onhour       0.0   0.41  0.41  0.39  0.17
12  USC00086323  onhour       0.0   0.40  0.40  0.18  0.11
18  USC00088841  onhour       NaN    NaN   NaN   NaN   NaN


In [72]:
# [orig cell 71 | env: precip | status: ok]
print(LG[(LG.grp=='metar')&(LG.best!=1)].round(2).to_string())
j=sidx[LG[(LG.best==-3)].id.iloc[0]]
print({lag:round(np.corrcoef(G[np.isfinite(Y[:,j])&np.isfinite(G[:,j]),j],Y[np.isfinite(Y[:,j])&np.isfinite(G[:,j]),j])[0,1],3) for lag,Y in Ys.items()}, Vst.iloc[j][['id','name']].tolist())

             id    grp src  best  rbest    r0    r1    r2
38  USW00012836  metar       NaN    NaN   NaN   NaN   NaN
46  USW00012850  metar       NaN    NaN   NaN   NaN   NaN
48  USW00012861  metar      -3.0   0.30  0.25  0.30  0.11
57  USW00012896  metar       NaN    NaN   NaN   NaN   NaN
68  USW00073805  metar       0.0   0.53  0.53  0.50  0.22
72  USW00092814  metar       0.0   0.62  0.62  0.52  0.21
{-3: np.float64(0.301), -2: np.float64(0.151), -1: np.float64(0.156), 0: np.float64(0.248), 1: np.float64(0.298), 2: np.float64(0.106), 3: np.float64(0.047)} ['USW00012861', 'OCALA JIM TAYLOR FLD']


In [73]:
%%bash
# [orig cell 72 | env: precip | status: ok]
mkdir -p prod/imerg && cat > prod/imerg_dl.py <<'EOF'
import requests, os, pandas as pd, concurrent.futures as cf, time, threading
class EDSession(requests.Session):
    def __init__(s,u,p): super().__init__(); s.auth=(u,p)
    def rebuild_auth(s, pr, resp):
        h=pr.headers
        if 'Authorization' in h:
            o=requests.utils.urlparse(resp.request.url).hostname; n=requests.utils.urlparse(pr.url).hostname
            if o!=n and 'urs.earthdata' not in (n or '') and 'urs.earthdata' not in (o or ''): del h['Authorization']
loc=threading.local()
def S():
    if not hasattr(loc,'s'): loc.s=EDSession(os.environ['EARTHDATA_USER'],os.environ['EARTHDATA_PASS'])
    return loc.s
hrs=pd.read_parquet('out/needed_hours.parquet').hour_start_utc
lon0,lon1,lat0,lat1=914,1010,1135,1219
ce=f'/Grid/precipitation[0:1:0][{lon0}:1:{lon1}][{lat0}:1:{lat1}]'
jobs=[]
for h in hrs:
    for m in (0,30):
        t=h+pd.Timedelta(minutes=m); e=t+pd.Timedelta(minutes=29,seconds=59)
        nm=f"3B-HHR.MS.MRG.3IMERG.{t:%Y%m%d}-S{t:%H%M%S}-E{e:%H%M%S}.{t.hour*60+t.minute:04d}.V07B.HDF5"
        jobs.append(nm)
base='https://opendap.earthdata.nasa.gov/collections/C2723754847-GES_DISC/granules/GPM_3IMERGHH.07%3A'
def get(nm):
    fn=f'prod/imerg/{nm}.nc4'
    if os.path.exists(fn): return 'skip'
    for a in range(4):
        try:
            r=S().get(base+nm+'.dap.nc4',params={'dap4.ce':ce},timeout=180)
            if r.status_code==404: return '404'
            if r.ok and r.content[:4]==b'\x89HDF': open(fn,'wb').write(r.content); return 'ok'
        except Exception: pass
        time.sleep(3*(a+1))
    return 'fail'
from collections import Counter
t0=time.time()
with cf.ThreadPoolExecutor(16) as ex: res=list(ex.map(get,jobs))
c=Counter(res); print(c, round((time.time()-t0)/60,1),'min')
open('prod/imerg_status.txt','w').write(str(c))
EOF
python prod/imerg_dl.py

Counter({'ok': 45111, '404': 377}) 61.5 min


In [74]:
# [orig cell 73 | env: precip | status: ok]
import h5py, concurrent.futures as cf
ref=xr.open_dataset('imerg_test.nc4',group='Grid'); la=ref.lat.values; lo=ref.lon.values
la2,lo2=np.meshgrid(la,lo,indexing='ij'); widx_i=common.window_index(la2,lo2); print('ok',sum(w['ok'] for w in widx_i))
def fname(t):
    e=t+pd.Timedelta(minutes=29,seconds=59)
    return f"prod/imerg/3B-HHR.MS.MRG.3IMERG.{t:%Y%m%d}-S{t:%H%M%S}-E{e:%H%M%S}.{t.hour*60+t.minute:04d}.V07B.HDF5.nc4"
def rd(fn):
    try:
        with h5py.File(fn,'r') as f:
            g=f['Grid'] if 'Grid' in f else f
            v=g['precipitation'][0].astype(np.float32).T   # (lat,lon)
            fv=g['precipitation'].attrs.get('_FillValue'); 
            v[v<0]=np.nan; return v
    except Exception as e: return None
def hour(k):
    h=Hneed[k]; a=rd(fname(h)); b=rd(fname(h+pd.Timedelta(minutes=30)))
    if a is None or b is None: return k,None
    return k,common.cut((a+b)/2,widx_i)
out=np.full((len(Hneed),len(Vst),8,8),np.nan,np.float32)
with cf.ThreadPoolExecutor(8) as ex:
    for k,a in ex.map(hour,range(len(Hneed))):
        if a is not None: out[k]=a
common.save('imerg',out,widx_i,Hneed)
print('hours with data',np.isfinite(out[:,:,3,3]).any(1).sum(), 'last', Hneed[np.where(np.isfinite(out[:,:,3,3]).any(1))[0].max()])
k=hpos[pd.Timestamp('2020-05-26 22:00')]; print('Miami 22Z', out[k,sidx['USW00012839'],2:6,2:6].round(1))
NBi,mi_=load('imerg'); print('imerg lag', lagcorr_g(NBi))

ok 79
hours with data 22555 last 2025-09-29 01:00:00
Miami 22Z [[ 1.8  1.8  1.1  0.1]
 [14.1  4.6  3.4  1.8]
 [12.3  8.8  4.8  3.2]
 [ 7.5  7.3  5.   3. ]]
imerg lag {'metar': {-2: np.float64(0.176), -1: np.float64(0.299), 0: np.float64(0.418), 1: np.float64(0.344), 2: np.float64(0.223)}, 'onhour': {-2: np.float64(0.199), -1: np.float64(0.327), 0: np.float64(0.402), 1: np.float64(0.306), 2: np.float64(0.199)}}


<kernel:40>:19: RuntimeWarning: All-NaN slice encountered
  NB[N+'max']=np.nanmax(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)
<kernel:40>:20: RuntimeWarning: Mean of empty slice
  NB[N+'mean']=np.nanmean(Am.reshape(A.shape[0],A.shape[1],-1),axis=2)

In [75]:
%%bash
# [orig cell 74 | env: precip | status: ok]
ps aux | grep -E "prod/(hourly_grib|cmorph|pccs)" | grep -v grep | awk '{print $3"%cpu", $4"%mem", $10, $13, $14}'; ls -la prod/*.npy

-rw-r--r--@ 1 mehdihatamigoloujeh  staff  459974784 Oct 18 21:59 prod/imerg_win.npy
-rw-r--r--@ 1 mehdihatamigoloujeh  staff  459974784 Oct 18 21:27 prod/merra2_win.npy
-rw-r--r--@ 1 mehdihatamigoloujeh  staff  459974784 Oct 18 21:41 prod/nldas2_win.npy


/bin/bash: /bin/ps: Operation not permitted

In [76]:
%%bash
# [orig cell 75 | env: precip | status: ok]
curl -s -o /dev/null -w 'stage4 %{http_code} %{size_download}B %{time_total}s\n' https://mesonet.agron.iastate.edu/archive/data/2019/07/01/stage4/ST4.2019070118.01h.grib
curl -s -o /dev/null -w 'mrms %{http_code} %{size_download}B %{time_total}s\n' https://mtarchive.geol.iastate.edu/2019/07/01/mrms/ncep/RadarOnly_QPE_01H/RadarOnly_QPE_01H_00.00_20190701-180000.grib2.gz
curl -s -o /dev/null -w 'cmorph %{http_code} %{size_download}B %{time_total}s\n' https://www.ncei.noaa.gov/data/cmorph-high-resolution-global-precipitation-estimates/access/30min/8km/2019/07/01/CMORPH_V1.0_ADJ_8km-30min_2019070118.nc
curl -s -o /dev/null -w 'pccs %{http_code} %{size_download}B %{time_total}s\n' https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly/2019/rgccs1h1918218.bin.gz

stage4 200 1009376B 12.083728s
mrms 200 513410B 2.300590s
cmorph 200 1713916B 2.163543s
pccs 200 1615116B 10.102402s


In [77]:
# [orig cell 76 | env: precip | status: ok]
w1=set()
for t in e20.peak_hour_utc.unique():
    for k in (-1,0,1): w1.add(t+pd.Timedelta(hours=k))
hrs_all=pd.read_csv('out/events_ge20_hours.csv',parse_dates=['hour_start_utc']).hour_start_utc
w1b=set(w1)
for t in hrs_all.unique():
    for k in (-1,0,1): w1b.add(t+pd.Timedelta(hours=k))
print(len(w1), len(w1b), len(Hneed))

12990 13425 22744


In [79]:
# [orig cell 78 | env: precip | status: ok]
import s3fs
fs3=s3fs.S3FileSystem(anon=True)
L=fs3.ls('nsf-ncar-era5/e5.oper.fc.sfc.accumu/202005/'); print(len(L)); print([x.split('/')[-1] for x in L if 'lsp' in x or 'cp.' in x or '_tp' in x][:6])
f=[x for x in L if '128_142_lsp' in x][0]
import h5py
with fs3.open(f,'rb',block_size=2**20) as fh:
    h=h5py.File(fh,'r'); print(list(h.keys())); d=h['LSP']; print(d.shape,d.chunks,d.compression, fs3.size(f)/1e6,'MB')
    print(h['forecast_initial_time'][:3], h['forecast_hour'][:])

76
['e5.oper.fc.sfc.accumu.128_050_lspf.ll025sc.2020050106_2020051606.nc', 'e5.oper.fc.sfc.accumu.128_050_lspf.ll025sc.2020051606_2020060106.nc', 'e5.oper.fc.sfc.accumu.128_142_lsp.ll025sc.2020050106_2020051606.nc', 'e5.oper.fc.sfc.accumu.128_142_lsp.ll025sc.2020051606_2020060106.nc', 'e5.oper.fc.sfc.accumu.128_143_cp.ll025sc.2020050106_2020051606.nc', 'e5.oper.fc.sfc.accumu.128_143_cp.ll025sc.2020051606_2020060106.nc']
['LSP', 'forecast_hour', 'forecast_initial_time', 'latitude', 'longitude', 'utc_date']
(30, 12, 721, 1440) (1, 12, 721, 1440) gzip 255.30125 MB
[1054806 1054818 1054830] [ 1  2  3  4  5  6  7  8  9 10 11 12]


In [80]:
# [orig cell 79 | env: precip | status: ok]
import cartopy.io.shapereader as shpreader, shapely, geopandas as gpd
from shapely.geometry import Point
coast=gpd.read_file(shpreader.natural_earth('10m','physical','coastline'))
urb=gpd.read_file(shpreader.natural_earth('10m','cultural','urban_areas'))
box=shapely.geometry.box(-89,23.5,-78,32)
coast=coast[coast.intersects(box)].to_crs(5070); urb=urb[urb.intersects(box)]
pts=gpd.GeoDataFrame(Vst[['id','name','lat','lon','source_group']],geometry=[Point(x,y) for x,y in zip(Vst.lon,Vst.lat)],crs=4326)
pts['coast_km']=pts.to_crs(5070).geometry.apply(lambda p: coast.distance(p).min()/1000)
pts['urban']=pts.geometry.apply(lambda p: urb.contains(p).any())
def region(r):
    if r.lat<25.2 and r.lon>-82.5: return 'Keys'
    if r.lon<-83.5 and r.lat>29.4: return 'Panhandle'
    if r.lat>=28.6: return 'North'
    if r.lat>=26.8: return 'Central'
    return 'South'
pts['region']=pts.apply(region,axis=1)
pts['coastal']=np.where(pts.coast_km<=10,'coastal (≤10 km)','inland (>10 km)')
print(pts.region.value_counts().to_dict(), pts.urban.value_counts().to_dict(), pts.coastal.value_counts().to_dict())
pts.drop(columns='geometry').to_csv('out/station_attributes.csv',index=False)

{'Central': 23, 'South': 18, 'North': 17, 'Panhandle': 17, 'Keys': 4} {False: 50, True: 29} {'inland (>10 km)': 40, 'coastal (≤10 km)': 39}


~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_urban_areas.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)

In [81]:
# [orig cell 80 | env: precip | status: ok]
from scipy import ndimage
cell_km2=(11.1*0.1)*(11.1*0.1)*np.cos(np.radians(la2))*100   # (lat,lon) area km2 per 0.1 deg cell
def hourfield(h):
    a=rd(fname(h)); b=rd(fname(h+pd.Timedelta(minutes=30)))
    return None if a is None or b is None else (a+b)/2
cache={}
def fld(h):
    if h not in cache: cache[h]=hourfield(h)
    return cache[h]
e20=e20.drop(columns=[c for c in ['obj_area_km2'] if c in e20])
ii0=np.array([w['i0'] for w in widx_i]); jj0=np.array([w['j0'] for w in widx_i])
def obj_area(row):
    fs_=[fld(row.peak_hour_utc+pd.Timedelta(hours=l)) for l in (-1,0,1)]
    fs_=[f for f in fs_ if f is not None]
    if not fs_: return np.nan
    F=np.nanmax(np.stack(fs_),0); lab,nl=ndimage.label(F>=1.0,structure=np.ones((3,3)))
    s=sidx[row.STATION]; i,j=ii0[s],jj0[s]
    ids=np.unique(lab[max(i-1,0):i+2,max(j-1,0):j+2]); ids=ids[ids>0]
    if len(ids)==0: return 0.0
    return float(cell_km2[np.isin(lab,ids)].sum())
t0=time.time(); e20['obj_area_km2']=[obj_area(r) for r in e20.itertuples()]; print(round(time.time()-t0),'s')
def stype(r):
    if r.is_TC: return 'Tropical cyclone'
    if r.peak_hour_utc.month in (11,12,1,2,3,4): return 'Cool-season (frontal/extratropical)'
    if np.isnan(r.obj_area_km2): return 'Unclassified'
    return 'Warm-season organized (MCS-like)' if r.obj_area_km2>=10000 else 'Warm-season isolated convection'
e20['storm_type']=e20.apply(stype,axis=1)
print(e20.storm_type.value_counts().to_dict())
for T in [30,50]: print(T, e20[e20.peak_mm_h>=T].storm_type.value_counts(normalize=True).round(2).to_dict())

22 s
{'Warm-season organized (MCS-like)': 4249, 'Cool-season (frontal/extratropical)': 1499, 'Warm-season isolated convection': 1242, 'Tropical cyclone': 428, 'Unclassified': 48}
30 {'Warm-season organized (MCS-like)': 0.61, 'Cool-season (frontal/extratropical)': 0.16, 'Warm-season isolated convection': 0.15, 'Tropical cyclone': 0.06, 'Unclassified': 0.01}
50 {'Warm-season organized (MCS-like)': 0.66, 'Cool-season (frontal/extratropical)': 0.13, 'Warm-season isolated convection': 0.13, 'Tropical cyclone': 0.07, 'Unclassified': 0.01}


In [82]:
# [orig cell 81 | env: precip | status: ok]
def obj_area2(row,thr=2.0,lags=(0,)):
    fs_=[fld(row.peak_hour_utc+pd.Timedelta(hours=l)) for l in lags]; fs_=[f for f in fs_ if f is not None]
    if not fs_: return np.nan
    F=np.nanmax(np.stack(fs_),0); lab,nl=ndimage.label(F>=thr,structure=np.ones((3,3)))
    s=sidx[row.STATION]; i,j=ii0[s],jj0[s]
    ids=np.unique(lab[max(i-1,0):i+2,max(j-1,0):j+2]); ids=ids[ids>0]
    return 0.0 if len(ids)==0 else float(cell_km2[np.isin(lab,ids)].sum())
ws=e20[e20.storm_type.str.startswith('Warm')].copy()
ws['a2']=[obj_area2(r) for r in ws.itertuples()]
print('area quantiles (peak hr, >=2mm/h) km2:', ws.a2.quantile([.1,.25,.5,.75,.9]).round(0).to_dict())
print('frac >=10k:', (ws.a2>=10000).mean().round(2), ' frac 0 (IMERG no rain>=2):',(ws.a2==0).mean().round(2))
# by local hour: organized share
lh=ws.peak_hour_utc.dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.hour
print(ws.assign(org=ws.a2>=10000,lh=lh).groupby(pd.cut(lh,[0,6,12,18,24],right=False)).org.mean().round(2).to_dict())

area quantiles (peak hr, >=2mm/h) km2: {0.1: 0.0, 0.25: 876.0, 0.5: 6382.0, 0.75: 20601.0, 0.9: 46338.0}
frac >=10k: 0.42  frac 0 (IMERG no rain>=2): 0.16
{Interval(0, 6, closed='left'): 0.46, Interval(6, 12, closed='left'): 0.5, Interval(12, 18, closed='left'): 0.39, Interval(18, 24, closed='left'): 0.43}


In [83]:
# [orig cell 82 | env: precip | status: ok]
attr=pts.drop(columns='geometry').set_index('id')
def label(e):
    e=e.copy()
    tc=e.apply(tcinfo,axis=1); e=pd.concat([e.drop(columns=[c for c in ['tc_dist_km','tc_name'] if c in e]),tc],axis=1)
    e['is_TC']=e.tc_dist_km<=500
    e['obj_area_km2']=[obj_area2(r) for r in e.itertuples()]
    lt=e.peak_hour_utc.dt.tz_localize('UTC').dt.tz_convert('America/New_York')
    e['local_hour']=lt.dt.hour; e['month']=e.peak_hour_utc.dt.month
    e['season']=np.where(e.month.between(5,10),'Wet (May–Oct)','Dry (Nov–Apr)')
    e['time_of_day']=pd.cut(e.local_hour,[0,6,12,18,24],right=False,labels=['Night 00–06','Morning 06–12','Afternoon 12–18','Evening 18–24']).astype(str)
    def st(r):
        if r.is_TC: return 'Tropical cyclone'
        if r.month in (11,12,1,2,3,4): return 'Cool-season (frontal)'
        return 'Warm-season organized (MCS-like)' if r.obj_area_km2>=10000 else 'Warm-season isolated convection'
    e['storm_type']=e.apply(st,axis=1)
    e['duration']=np.where(e.n_exceed_hours>1,'multi-hour','single-hour')
    e=e.join(attr[['region','coastal','urban','coast_km']],on='STATION'); e['urban']=np.where(e.urban,'urban','rural')
    return e
for T in EV:
    EV[T]=label(EV[T]); EV[T].to_csv(f'out/events_ge{T}_labeled.csv',index=False)
    print(T, EV[T].storm_type.value_counts().to_dict())

20 {'Warm-season isolated convection': 3232, 'Warm-season organized (MCS-like)': 2307, 'Cool-season (frontal)': 1499, 'Tropical cyclone': 428}
30 {'Warm-season isolated convection': 1287, 'Warm-season organized (MCS-like)': 1034, 'Cool-season (frontal)': 488, 'Tropical cyclone': 187}
40 {'Warm-season isolated convection': 505, 'Warm-season organized (MCS-like)': 418, 'Cool-season (frontal)': 172, 'Tropical cyclone': 74}
50 {'Warm-season isolated convection': 208, 'Warm-season organized (MCS-like)': 178, 'Cool-season (frontal)': 65, 'Tropical cyclone': 34}


> Original cell 83 ended with status **error** in the session; it is kept for the record.

In [84]:
%%bash
# [orig cell 83 | env: precip | status: error]
python prod/hourly_grib.py mrms_radaronly 2>&1 | tail -20

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

> Original cell 84 ended with status **error** in the session; it is kept for the record.

In [85]:
%%bash
# [orig cell 84 | env: precip | status: error]
python prod/hourly_grib.py mrms_gaugecorr 2>&1 | tail -20

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [87]:
# [orig cell 86 | env: precip | status: ok]
H1=sorted(w1b); pd.Series(H1).to_frame('hour_start_utc').to_parquet('out/needed_hours_pm1.parquet'); print(len(H1))
script=r'''
import sys, os, gzip, time, json, numpy as np, pandas as pd, requests, eccodes, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
prod=sys.argv[1]; NPROC=int(sys.argv[2])
H=hours(); n=len(H)
H1=set(pd.read_parquet('out/needed_hours_pm1.parquet').hour_start_utc)
todo=[k for k in range(n) if H[k] in H1]
def url(h):
    t=h+pd.Timedelta(hours=1)
    if prod=='mrms_radaronly':
        return [f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/RadarOnly_QPE_01H/RadarOnly_QPE_01H_00.00_{t:%Y%m%d-%H}0000.grib2.gz']
    return [f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/MultiSensor_QPE_01H_Pass2/MultiSensor_QPE_01H_Pass2_00.00_{t:%Y%m%d-%H}0000.grib2.gz',
            f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/GaugeCorr_QPE_01H/GaugeCorr_QPE_01H_00.00_{t:%Y%m%d-%H}0000.grib2.gz']
# MRMS regular grid: lat 54.995 -> 20.005 step 0.01, lon -129.995 -> -60.005
lat=54.995-0.01*np.arange(3500); lon=-129.995+0.01*np.arange(7000)
ii=np.where((lat>23.5)&(lat<31.5))[0]; jj=np.where((lon>-88.5)&(lon<-79.5))[0]
la2,lo2=np.meshgrid(lat[ii],lon[jj],indexing='ij'); WIDX=window_index(la2,lo2)
def work(k):
    h=H[k]
    for u in url(h):
        for a in range(3):
            try:
                r=requests.get(u,timeout=(20,60))
                if r.status_code==404: break
                if r.ok:
                    b=gzip.decompress(r.content); gid=eccodes.codes_new_from_message(b)
                    v=eccodes.codes_get_values(gid).reshape(3500,7000)[ii[0]:ii[-1]+1,jj[0]:jj[-1]+1].astype(np.float32); eccodes.codes_release(gid)
                    v[(v<0)|(v>9000)]=np.nan
                    return k,cut(v,WIDX),u.split('/')[-1][:12]
            except Exception: time.sleep(2)
    return k,None,''
if __name__=='__main__':
    os.makedirs(f'prod/ck_{prod}',exist_ok=True)
    chunks=[todo[i:i+500] for i in range(0,len(todo),500)]
    t0=time.time()
    with cf.ProcessPoolExecutor(NPROC) as ex:
        for ci,ch in enumerate(chunks):
            fn=f'prod/ck_{prod}/c{ci:03d}.npz'
            if os.path.exists(fn): continue
            ks=[];arrs=[];srcs=[]
            for k,a,s in ex.map(work,ch):
                if a is not None: ks.append(k);arrs.append(a);srcs.append(s)
            np.savez(fn,k=np.array(ks),a=np.array(arrs) if arrs else np.zeros((0,len(WIDX),W,W),np.float32),s=np.array(srcs))
            open(f'prod/{prod}_progress.txt','w').write(f'{ci+1}/{len(chunks)} chunks {round((time.time()-t0)/60,1)} min\n')
    out=np.full((n,len(WIDX),W,W),np.nan,np.float32); src=['']*n
    for ci in range(len(chunks)):
        z=np.load(f'prod/ck_{prod}/c{ci:03d}.npz')
        for k,a,s in zip(z['k'],z['a'],z['s']): out[k]=a; src[k]=str(s)
    save(prod,out,WIDX,H,extra=dict(source=src))
    print(prod,'done; hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',len(todo),round((time.time()-t0)/60,1),'min')
'''
open('prod/mrms_mp.py','w').write(script)

13425


In [88]:
%%bash
# [orig cell 87 | env: precip | status: ok]
python prod/mrms_mp.py mrms_gaugecorr 4 2>&1 | tail -3

    nsems_max = os.sysconf("SC_SEM_NSEMS_MAX")
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
PermissionError: [Errno 1] Operation not permitted


In [89]:
%%bash
# [orig cell 88 | env: precip | status: ok]
python prod/mrms_mp.py mrms_radaronly 4 2>&1 | tail -3

    nsems_max = os.sysconf("SC_SEM_NSEMS_MAX")
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
PermissionError: [Errno 1] Operation not permitted


In [90]:
%%bash
# [orig cell 89 | env: precip | status: ok]
cd prod && python - <<'EOF'
s=open('mrms_mp.py').read()
s=s.replace("prod=sys.argv[1]; NPROC=int(sys.argv[2])","prod=sys.argv[1]; SHARD=int(sys.argv[2]); NSH=int(sys.argv[3])")
old_main=s[s.index("if __name__=='__main__':"):]
new_main='''if __name__=='__main__':
    os.makedirs(f'prod/ck_{prod}',exist_ok=True)
    chunks=[todo[i:i+250] for i in range(0,len(todo),250)]
    t0=time.time()
    if SHARD>=0:
        with cf.ThreadPoolExecutor(3) as ex:
            for ci,ch in enumerate(chunks):
                if ci%NSH!=SHARD: continue
                fn=f'prod/ck_{prod}/c{ci:03d}.npz'
                if os.path.exists(fn): continue
                ks=[];arrs=[];srcs=[]
                for k,a,s_ in ex.map(work,ch):
                    if a is not None: ks.append(k);arrs.append(a);srcs.append(s_)
                np.savez(fn+'.tmp.npz',k=np.array(ks),a=np.array(arrs) if arrs else np.zeros((0,len(WIDX),W,W),np.float32),s=np.array(srcs)); os.replace(fn+'.tmp.npz',fn)
                open(f'prod/{prod}_progress_{SHARD}.txt','w').write(f'chunk {ci} of {len(chunks)} {round((time.time()-t0)/60,1)} min\\n')
    else:
        out=np.full((n,len(WIDX),W,W),np.nan,np.float32); src=['']*n; miss=0
        for ci in range(len(chunks)):
            fn=f'prod/ck_{prod}/c{ci:03d}.npz'
            if not os.path.exists(fn): miss+=1; continue
            z=np.load(fn)
            for k,a,s_ in zip(z['k'],z['a'],z['s']): out[k]=a; src[k]=str(s_)
        save(prod,out,WIDX,H,extra=dict(source=src))
        print(prod,'merged; missing chunks',miss,'hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',len(todo))
'''
open('mrms_mp.py','w').write(s.replace(old_main,new_main))
EOF
grep -n "SHARD" prod/mrms_mp.py | head -3

grep: prod/mrms_mp.py: No such file or directory

In [91]:
%%bash
# [orig cell 90 | env: precip | status: ok]
cat prod/*_progress_*.txt 2>/dev/null; ls prod/ck_mrms_radaronly prod/ck_mrms_gaugecorr 2>/dev/null | wc -l; tail -2 prod/log_ro_0.txt prod/log_gc_0.txt

chunk 0 of 54 6.1 min
       4
==> prod/log_ro_0.txt <==

==> prod/log_gc_0.txt <==


> Original cell 91 ended with status **error** in the session; it is kept for the record.

In [92]:
%%bash
# [orig cell 91 | env: precip | status: error]
python prod/hourly_grib.py stage4 2>&1 | tail -20

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

> Original cell 92 ended with status **error** in the session; it is kept for the record.

In [93]:
%%bash
# [orig cell 92 | env: precip | status: error]
cat > prod/pccs.py <<'EOF'
import sys, os, gzip, time, numpy as np, pandas as pd, requests, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
H=hours(); n=len(H)
B='https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly/'
lat_p=59.98-0.04*np.arange(3000); lon_p=0.02+0.04*np.arange(9000)
ii=np.where((lat_p>23.0)&(lat_p<32.5))[0]; jj=np.where((lon_p>360-89.5)&(lon_p<360-78.5))[0]
la2,lo2=np.meshgrid(lat_p[ii],lon_p[jj]-360,indexing='ij'); widx=window_index(la2,lo2); print('ok st',sum(w['ok'] for w in widx),flush=True)
def get(k):
    h=H[k]; u=f'{B}{h:%Y}/rgccs1h{h:%y}{h.dayofyear:03d}{h:%H}.bin.gz'
    for a in range(4):
        try:
            r=requests.get(u,timeout=180)
            if r.status_code==404: return k,None
            if r.ok:
                x=np.frombuffer(gzip.decompress(r.content),'>i2').reshape(3000,9000)[ii[0]:ii[-1]+1,jj[0]:jj[-1]+1].astype(np.float32)
                x[x<0]=np.nan; return k,cut(x/100.,widx)
        except Exception: time.sleep(3*(a+1))
    return k,None
out=np.full((n,len(widx),W,W),np.nan,np.float32); t0=time.time()
with cf.ThreadPoolExecutor(10) as ex:
    for i,(k,a) in enumerate(ex.map(get,range(n))):
        if a is not None: out[k]=a
        if (i+1)%3000==0: print(i+1,round((time.time()-t0)/60,1),'min',flush=True)
save('pccs',out,widx,H)
print('pccs done; hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',n,round((time.time()-t0)/60,1),'min')
EOF
python prod/pccs.py 2>&1 | tail -8

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [95]:
%%bash
# [orig cell 94 | env: precip | status: ok]
cat > prod/shard_dl.py <<'EOF'
import sys, os, gzip, time, numpy as np, pandas as pd, requests, eccodes, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
prod=sys.argv[1]; SHARD=int(sys.argv[2]); NSH=int(sys.argv[3]); NT=int(sys.argv[4]) if len(sys.argv)>4 else 3
H=hours(); n=len(H)
H1=set(pd.read_parquet('out/needed_hours_pm1.parquet').hour_start_utc)
todo=[k for k in range(n) if H[k] in H1]
if prod=='stage4':
    import json
    def url(h):
        t=h+pd.Timedelta(hours=1); return f'https://mesonet.agron.iastate.edu/archive/data/{t:%Y/%m/%d}/stage4/ST4.{t:%Y%m%d%H}.01h.grib'
    def grid():
        r=requests.get(url(pd.Timestamp('2020-05-26 22:00')),timeout=60); gid=eccodes.codes_new_from_message(r.content)
        ni=eccodes.codes_get(gid,'Ni'); nj=eccodes.codes_get(gid,'Nj')
        la=eccodes.codes_get_array(gid,'latitudes').reshape(nj,ni); lo=eccodes.codes_get_array(gid,'longitudes').reshape(nj,ni); lo=np.where(lo>180,lo-360,lo)
        eccodes.codes_release(gid); return la,lo
    def decode(b):
        gid=eccodes.codes_new_from_message(b); ni=eccodes.codes_get(gid,'Ni'); nj=eccodes.codes_get(gid,'Nj')
        v=eccodes.codes_get_values(gid).reshape(nj,ni).astype(np.float32); mv=eccodes.codes_get(gid,'missingValue'); eccodes.codes_release(gid)
        v[v==mv]=np.nan; v[v<0]=np.nan; return v
elif prod=='pccs':
    lat_p=59.98-0.04*np.arange(3000); lon_p=0.02+0.04*np.arange(9000)
    ii=np.where((lat_p>23.0)&(lat_p<32.5))[0]; jj=np.where((lon_p>360-89.5)&(lon_p<360-78.5))[0]
    def url(h): return f'https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly/{h:%Y}/rgccs1h{h:%y}{h.dayofyear:03d}{h:%H}.bin.gz'
    def grid(): return np.meshgrid(lat_p[ii],lon_p[jj]-360,indexing='ij')
    def decode(b):
        x=np.frombuffer(gzip.decompress(b),'>i2').reshape(3000,9000)[ii[0]:ii[-1]+1,jj[0]:jj[-1]+1].astype(np.float32); x[x<0]=np.nan; return x/100.
la2,lo2=grid(); WIDX=window_index(la2,lo2)
def work(k):
    for a in range(3):
        try:
            r=requests.get(url(H[k]),timeout=(20,90))
            if r.status_code==404: return k,None
            if r.ok: return k,cut(decode(r.content),WIDX)
        except Exception: time.sleep(2)
    return k,None
os.makedirs(f'prod/ck_{prod}',exist_ok=True)
chunks=[todo[i:i+250] for i in range(0,len(todo),250)]; t0=time.time()
if SHARD>=0:
    with cf.ThreadPoolExecutor(NT) as ex:
        for ci,ch in enumerate(chunks):
            if ci%NSH!=SHARD: continue
            fn=f'prod/ck_{prod}/c{ci:03d}.npz'
            if os.path.exists(fn): continue
            ks=[];arrs=[]
            for k,a in ex.map(work,ch):
                if a is not None: ks.append(k);arrs.append(a)
            np.savez(fn+'.tmp.npz',k=np.array(ks),a=np.array(arrs) if arrs else np.zeros((0,len(WIDX),W,W),np.float32)); os.replace(fn+'.tmp.npz',fn)
            open(f'prod/{prod}_progress_{SHARD}.txt','w').write(f'chunk {ci} of {len(chunks)} {round((time.time()-t0)/60,1)} min\n')
else:
    out=np.full((n,len(WIDX),W,W),np.nan,np.float32); miss=0
    for ci in range(len(chunks)):
        fn=f'prod/ck_{prod}/c{ci:03d}.npz'
        if not os.path.exists(fn): miss+=1; continue
        z=np.load(fn)
        for k,a in zip(z['k'],z['a']): out[k]=a
    save(prod,out,WIDX,H)
    print(prod,'merged; missing chunks',miss,'hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',len(todo))
EOF
echo ok

ok


In [96]:
%%bash
# [orig cell 95 | env: precip | status: ok]
for p in mrms_radaronly mrms_gaugecorr stage4 pccs; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l) chunks done of 54 | $(cat prod/${p}_progress_0.txt 2>/dev/null)"; done; tail -c 300 prod/log_st4_0.txt prod/log_pccs_0.txt

mrms_radaronly:       16 chunks done of 54 | chunk 12 of 54 38.6 min
mrms_gaugecorr:       16 chunks done of 54 | chunk 12 of 54 33.4 min
stage4:        0 chunks done of 54 | 
pccs:        0 chunks done of 54 | 
==> prod/log_st4_0.txt <==

==> prod/log_pccs_0.txt <==


> Original cell 96 ended with status **error** in the session; it is kept for the record.

In [97]:
%%bash
# [orig cell 96 | env: precip | status: error]
for p in mrms_radaronly mrms_gaugecorr stage4 pccs; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54 | $(cat prod/${p}_progress_*.txt 2>/dev/null | tr '\n' ';')"; done; ls -la prod/cmorph_win.npy 2>/dev/null

mrms_radaronly:       25/54 | chunk 20 of 54 60.3 min;chunk 21 of 54 59.4 min;chunk 22 of 54 60.8 min;chunk 27 of 54 66.5 min;
mrms_gaugecorr:       29/54 | chunk 28 of 54 69.4 min;chunk 25 of 54 68.7 min;chunk 26 of 54 68.2 min;chunk 27 of 54 65.4 min;
stage4:        4/54 | chunk 0 of 54 23.2 min;chunk 1 of 54 23.4 min;chunk 2 of 54 24.2 min;chunk 3 of 54 23.7 min;
pccs:        4/54 | chunk 0 of 54 20.3 min;chunk 1 of 54 18.8 min;chunk 2 of 54 17.3 min;chunk 3 of 54 17.2 min;


In [98]:
%%bash
# [orig cell 97 | env: precip | status: ok]
curl -s -o /dev/null -w 'stage4 %{size_download}B %{time_total}s %{time_starttransfer}s-ttfb\n' https://mesonet.agron.iastate.edu/archive/data/2018/07/01/stage4/ST4.2018070118.01h.grib
curl -s -o /dev/null -w 'pccs %{size_download}B %{time_total}s %{time_starttransfer}s-ttfb\n' https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly/2018/rgccs1h1818218.bin.gz
curl -s -o /dev/null -w 'mrms %{size_download}B %{time_total}s %{time_starttransfer}s-ttfb\n' https://mtarchive.geol.iastate.edu/2018/07/01/mrms/ncep/RadarOnly_QPE_01H/RadarOnly_QPE_01H_00.00_20180701-180000.grib2.gz

stage4 1078652B 21.408989s 0.529204s-ttfb
pccs 1898476B 39.893584s 0.572476s-ttfb
mrms 743023B 8.402796s 1.067930s-ttfb


In [99]:
# [orig cell 98 | env: precip | status: ok]
inH1=Hneed.isin(set(H1)).values
LAG={'nldas2':1}
def load_shift(name):
    NB,meta=load(name); L=LAG.get(name,0)
    if L: NB={k:shift(v,L) for k,v in NB.items()}
    return NB,meta
def score2(name,NB,boot=200):
    rng=np.random.default_rng(0); rows=[]; per=[]
    sh={(N,l):shift(NB[N+'max'],l) for N in ['N1','N4','N9','N16'] for l in (-1,0,1)}
    Gc=G.copy(); Gc[~inH1]=np.nan
    for T,e in EV.items():
        e=e[(e.flag=='')&e.k.notna()]; k=e.k.astype(int).values; s=e.s.values; g=e.peak_mm_h.values
        for N in ['N1','N4','N9','N16']:
            v0=sh[(N,0)][k,s]; vt=np.nanmax(np.stack([sh[(N,l)][k,s] for l in (-1,0,1)]),0); vm=NB[N+'mean'][k,s]
            ok=np.isfinite(v0); st=s[ok]; hit=(v0[ok]>=T).astype(float)
            us=np.unique(st); grp_={u:np.where(st==u)[0] for u in us}
            bs=[hit[np.concatenate([grp_[u] for u in rng.choice(us,len(us))])].mean() for _ in range(boot)]
            Y=sh[(N,0)]; okc=np.isfinite(Y)&np.isfinite(Gc); o=Gc[okc]>=T; f=Y[okc]>=T
            a=(o&f).sum(); b_=(f&~o).sum(); c=(o&~f).sum()
            rows.append(dict(product=name,threshold=T,nbhd=N,n_events=int(ok.sum()),
                hit_strict_0h=hit.mean(),hit_lo=np.quantile(bs,.025),hit_hi=np.quantile(bs,.975),hit_strict_pm1h=np.mean(vt[ok]>=T),
                hit_ge10=np.mean(v0[ok]>=10),hit_ge1=np.mean(v0[ok]>=1),
                ratio_med=np.median(v0[ok]/g[ok]),ratio_q25=np.quantile(v0[ok]/g[ok],.25),ratio_q75=np.quantile(v0[ok]/g[ok],.75),ratio_nbmean_med=np.nanmedian(vm[ok]/g[ok]),
                POD_h=a/(a+c) if a+c else np.nan,FAR_h=b_/(a+b_) if a+b_ else np.nan,CSI_h=a/(a+b_+c) if a+b_+c else np.nan,FBI_h=(a+b_)/(a+c) if a+c else np.nan,n_hours=int(okc.sum())))
            pe=e[ok].copy(); pe['product']=name; pe['threshold']=T; pe['nbhd']=N; pe['prod_0h']=v0[ok]; pe['prod_pm1h']=vt[ok]; pe['prod_nbmean']=vm[ok]
            per.append(pe)
    return pd.DataFrame(rows),pd.concat(per)
SC={};PE={}
for nm in ['imerg','merra2','nldas2']:
    NB,_=load_shift(nm); SC[nm],PE[nm]=score2(nm,NB)
    print(nm, SC[nm][SC[nm].nbhd.isin(['N1','N16'])][['threshold','nbhd','n_events','hit_strict_0h','hit_strict_pm1h','hit_ge10','ratio_med','FAR_h','CSI_h']].round(2).to_string(index=False))

imerg  threshold nbhd  n_events  hit_strict_0h  hit_strict_pm1h  hit_ge10  ratio_med  FAR_h  CSI_h
        20   N1      7348           0.02             0.03      0.13       0.12   0.70   0.02
        20  N16      7348           0.10             0.14      0.35       0.25   0.81   0.07
        30   N1      2917           0.01             0.01      0.15       0.10   0.78   0.01
        30  N16      2917           0.03             0.05      0.40       0.21   0.86   0.03
        40   N1      1107           0.00             0.00      0.19       0.09   0.83   0.00
        40  N16      1107           0.01             0.01      0.45       0.18   0.93   0.01
        50   N1       464           0.00             0.00      0.19       0.08   1.00   0.00
        50  N16       464           0.00             0.00      0.49       0.16   0.92   0.00
merra2  threshold nbhd  n_events  hit_strict_0h  hit_strict_pm1h  hit_ge10  ratio_med  FAR_h  CSI_h
        20   N1      7407           0.00             0.00

In [100]:
%%bash
# [orig cell 99 | env: precip | status: ok]
for p in mrms_radaronly mrms_gaugecorr stage4 pccs; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54"; done

mrms_radaronly:       41/54
mrms_gaugecorr:       44/54
stage4:       11/54
pccs:       12/54


In [101]:
%%bash
# [orig cell 100 | env: precip | status: ok]
for s in 0 1 2 3; do python prod/mrms_mp.py mrms_gaugecorr $s 4 > prod/log_gc_$s.txt 2>&1 & done; wait; python prod/mrms_mp.py mrms_gaugecorr -1 4 2>&1 | tail -2

mrms_gaugecorr merged; missing chunks 0 hours with data 13220 of 13425


In [102]:
# [orig cell 101 | env: precip | status: ok]
import warnings; warnings.filterwarnings('ignore')
NBg,mg_=load('mrms_gaugecorr'); print('mrms_gc lag',lagcorr_g(NBg))
src=pd.Series(mg_['extra']['source']); print(src.str[:8].value_counts().to_dict())

mrms_gc lag {'metar': {-2: np.float64(0.144), -1: np.float64(0.314), 0: np.float64(0.866), 1: np.float64(0.295), 2: np.float64(0.133)}, 'onhour': {-2: np.float64(0.176), -1: np.float64(0.463), 0: np.float64(0.731), 1: np.float64(0.205), 2: np.float64(0.116)}}
{'': 9524, 'GaugeCor': 7025, 'MultiSen': 6195}


In [103]:
%%bash
# [orig cell 102 | env: precip | status: ok]
grep -c "SHARD" prod/mrms_mp.py; for s in 0 1 2 3; do python prod/mrms_mp.py mrms_radaronly $s 4 > prod/log_ro_$s.txt 2>&1 & done; wait; python prod/mrms_mp.py mrms_radaronly -1 4 2>&1 | tail -2

4
mrms_radaronly merged; missing chunks 0 hours with data 13386 of 13425


In [104]:
# [orig cell 103 | env: precip | status: ok]
NBr,mr_=load('mrms_radaronly'); print('mrms_ro lag',lagcorr_g(NBr))

mrms_ro lag {'metar': {-2: np.float64(0.149), -1: np.float64(0.333), 0: np.float64(0.825), 1: np.float64(0.266), 2: np.float64(0.133)}, 'onhour': {-2: np.float64(0.176), -1: np.float64(0.46), 0: np.float64(0.698), 1: np.float64(0.201), 2: np.float64(0.118)}}


In [105]:
%%bash
# [orig cell 104 | env: precip | status: ok]
for p in stage4 pccs; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54"; done; ls prod/cmorph_win.npy 2>&1 | tail -1

stage4:       39/54
pccs:       24/54
ls: prod/cmorph_win.npy: No such file or directory


> Original cell 105 ended with status **error** in the session; it is kept for the record.

In [106]:
%%bash
# [orig cell 105 | env: precip | status: error]
cat > prod/cmorph.py <<'EOF'
import sys, os, io, time, numpy as np, pandas as pd, requests, xarray as xr, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
H=hours(); n=len(H)
B='https://www.ncei.noaa.gov/data/cmorph-high-resolution-global-precipitation-estimates/access/30min/8km/'
# subset box indices from a reference file
ref=xr.open_dataset('cm_test.nc'); la=ref.lat.values; lo=ref.lon.values
ii=np.where((la>23.0)&(la<32.5))[0]; jj=np.where((lo>360-89.5)&(lo<360-78.5))[0]
la2,lo2=np.meshgrid(la[ii],lo[jj]-360,indexing='ij'); widx=window_index(la2,lo2); print('ok st',sum(w['ok'] for w in widx),flush=True)
def get(k):
    h=H[k]; u=f'{B}{h:%Y/%m/%d}/CMORPH_V1.0_ADJ_8km-30min_{h:%Y%m%d%H}.nc'   # file HH holds HH:00 and HH:30 half-hours
    for a in range(4):
        try:
            r=requests.get(u,timeout=180)
            if r.status_code==404: return k,None
            if r.ok:
                ds=xr.open_dataset(io.BytesIO(r.content),engine='h5netcdf')
                v=ds.cmorph.values[:,ii[0]:ii[-1]+1,jj[0]:jj[-1]+1]
                return k,cut(np.nanmean(v,0) if np.isfinite(v).all(0).any() else v.mean(0),widx)   # mean of two 30-min rates = mm in hour
        except Exception: time.sleep(3*(a+1))
    return k,None
out=np.full((n,len(widx),W,W),np.nan,np.float32); t0=time.time()
with cf.ThreadPoolExecutor(12) as ex:
    for i,(k,a) in enumerate(ex.map(get,range(n))):
        if a is not None: out[k]=a
        if (i+1)%3000==0: print(i+1,round((time.time()-t0)/60,1),'min',flush=True)
save('cmorph',out,widx,H)
print('cmorph done; hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',n,round((time.time()-t0)/60,1),'min')
EOF
python prod/cmorph.py 2>&1 | grep -v Warning | tail -8

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [108]:
# [orig cell 107 | env: precip | status: ok]
import netCDF4
b=open('cm_test.nc','rb').read(); d=netCDF4.Dataset('mem.nc',memory=b); la_=d['lat'][:].data; lo_=d['lon'][:].data
i=np.argmin(abs(la_-25.8)); j=np.argmin(abs(lo_-(360-80.3))); print(np.ma.filled(d['cmorph'][:,i,j],np.nan), d['cmorph'][:,i,j].mean())
import time; time.sleep(60)
print(open('prod/log_cm_0.txt').read()[-400:])

[26.23 27.8 ] 27.015
e accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = 'mem.nc', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0
    major: File accessibility
    minor: Unable to open file



> Original cell 108 ended with status **error** in the session; it is kept for the record.

In [109]:
%%bash
# [orig cell 108 | env: precip | status: error]
sleep 120; grep -c "HDF5-DIAG" prod/log_cm_0.txt; grep -E "Traceback|Error:" prod/log_cm_*.txt | grep -v "H5\|major\|minor" | head; ls prod/ck_cmorph | wc -l; cat prod/cmorph_progress_* 2>/dev/null

67
       0


> Original cell 109 ended with status **error** in the session; it is kept for the record.

In [110]:
%%bash
# [orig cell 109 | env: precip | status: error]
python - <<'EOF'
s=open('prod/shard_dl.py').read()
add='''elif prod=='cmorph':
    import netCDF4
    _r=netCDF4.Dataset('cm_test.nc'); _la=_r['lat'][:].data; _lo=_r['lon'][:].data; _r.close()
    ii=np.where((_la>23.0)&(_la<32.5))[0]; jj=np.where((_lo>360-89.5)&(_lo<360-78.5))[0]
    def url(h): return f'https://www.ncei.noaa.gov/data/cmorph-high-resolution-global-precipitation-estimates/access/30min/8km/{h:%Y/%m/%d}/CMORPH_V1.0_ADJ_8km-30min_{h:%Y%m%d%H}.nc'
    def grid(): return np.meshgrid(_la[ii],_lo[jj]-360,indexing='ij')
    def decode(b):
        d=netCDF4.Dataset('mem.nc',memory=b); v=d['cmorph'][:,ii[0]:ii[-1]+1,jj[0]:jj[-1]+1]; d.close()
        v=np.ma.filled(v.astype(np.float32),np.nan); return v.mean(0)
la2,lo2=grid()'''
s=s.replace("la2,lo2=grid()",add,1)
open('prod/shard_dl.py','w').write(s)
EOF
for s in 0 1 2 3; do python prod/shard_dl.py cmorph $s 4 3 > prod/log_cm_$s.txt 2>&1 & done; wait; python prod/shard_dl.py cmorph -1 4 2>&1 | tail -2; tail -3 prod/log_cm_0.txt

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [112]:
# [orig cell 111 | env: precip | status: ok]
import h5py, io
with h5py.File(io.BytesIO(b),'r') as f:
    v=f['cmorph']; print(v.shape, dict(v.attrs).keys())
    x=v[:,i,j]; print(x, v.attrs.get('scale_factor'), v.attrs.get('add_offset'), v.attrs.get('_FillValue'))

(2, 1649, 4948) dict_keys(['standard_name', 'long_name', 'units', 'missing_value', 'valid_min', 'valid_max', 'scale_factor', 'coordinates', 'comment', 'DIMENSION_LIST'])
[2623 2780] [0.01] None None


In [114]:
%%bash
# [orig cell 113 | env: precip | status: ok]
for s in 0 1 2 3; do python prod/shard_dl.py stage4 $s 4 4 > prod/log_st4_$s.txt 2>&1 & done; wait; python prod/shard_dl.py stage4 -1 4 2>&1 | tail -2

stage4 merged; missing chunks 0 hours with data 13415 of 13425


> Original cell 114 ended with status **error** in the session; it is kept for the record.

In [115]:
%%bash
# [orig cell 114 | env: precip | status: error]
for p in pccs cmorph; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54 $(cat prod/${p}_progress_0.txt 2>/dev/null)"; done; grep -c DIAG prod/log_cm_0.txt

pccs:       28/54 chunk 24 of 54 130.3 min
cmorph:        0/54 
0


In [116]:
# [orig cell 115 | env: precip | status: ok]
NBs,ms_=load('stage4'); print('stage4 lag',lagcorr_g(NBs)); print('ok stations',sum(w['ok'] for w in ms_['widx']))

stage4 lag {'metar': {-2: np.float64(0.16), -1: np.float64(0.353), 0: np.float64(0.798), 1: np.float64(0.275), 2: np.float64(0.135)}, 'onhour': {-2: np.float64(0.19), -1: np.float64(0.477), 0: np.float64(0.698), 1: np.float64(0.209), 2: np.float64(0.121)}}
ok stations 79


In [117]:
%%bash
# [orig cell 116 | env: precip | status: ok]
for p in pccs cmorph; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54 $(cat prod/${p}_progress_*.txt 2>/dev/null | tr '\n' ';')"; done

pccs:       36/54 chunk 32 of 54 155.6 min;chunk 33 of 54 157.5 min;chunk 34 of 54 155.4 min;chunk 35 of 54 154.7 min;
cmorph:       24/54 chunk 20 of 54 37.3 min;chunk 21 of 54 38.2 min;chunk 22 of 54 37.9 min;chunk 23 of 54 37.5 min;


In [118]:
# [orig cell 117 | env: precip | status: ok]
PRODS={'mrms_gaugecorr':'MRMS gauge-corrected (1 km)','mrms_radaronly':'MRMS radar-only (1 km)','stage4':'Stage IV (4 km)','nldas2':'NLDAS-2 (0.125°)',
       'pccs':'PERSIANN-CCS (0.04°)','cmorph':'CMORPH CDR (8 km)','imerg':'IMERG V07 Final (0.1°)','merra2':'MERRA-2 (0.5°×0.625°)'}
PTYPE={'mrms_gaugecorr':'radar+gauge','mrms_radaronly':'radar','stage4':'radar+gauge','nldas2':'gauge/radar-based reanalysis forcing','pccs':'satellite (IR)','cmorph':'satellite (MW+IR, bias-corrected)','imerg':'satellite (MW+IR, gauge-adjusted)','merra2':'atmospheric reanalysis'}
# common event set: events where ALL available products have data at N1 lag0 (computed later), plus per-product set
def run(prods):
    SCs=[];PEs=[];NBall={}
    for nm in prods:
        NB,meta=load_shift(nm); NBall[nm]=(NB,meta)
        sc,pe=score2(nm,NB,boot=300); sc['label']=PRODS[nm]; SCs.append(sc); PEs.append(pe)
    return pd.concat(SCs),pd.concat(PEs),NBall
avail=[p for p in PRODS if os.path.exists(f'prod/{p}_win.npy')]; print(avail)
SCa,PEa,NBall=run(avail)
piv=SCa[SCa.nbhd=='N1'].pivot_table(index='product',columns='threshold',values='hit_strict_0h').round(3); print(piv)

['mrms_gaugecorr', 'mrms_radaronly', 'stage4', 'nldas2', 'imerg', 'merra2']
threshold          20     30     40     50
product                                   
imerg           0.020  0.005  0.002  0.000
merra2          0.001  0.000  0.000  0.000
mrms_gaugecorr  0.530  0.428  0.350  0.330
mrms_radaronly  0.459  0.357  0.254  0.186
nldas2          0.171  0.085  0.050  0.026
stage4          0.439  0.346  0.278  0.239


In [119]:
%%bash
# [orig cell 118 | env: precip | status: ok]
for p in pccs cmorph; do echo "$p: $(ls prod/ck_$p 2>/dev/null | grep -v tmp | wc -l)/54"; done

pccs:       46/54
cmorph:       53/54


In [120]:
%%bash
# [orig cell 119 | env: precip | status: ok]
rm -rf prod/ck_cmorph; python - <<'EOF'
s=open('prod/shard_dl.py').read()
old=s[s.index("    def decode(b):\n        d=netCDF4"):s.index("la2,lo2=grid()",s.index("prod=='cmorph'"))]
new='''    import h5py, io
    def decode(b):
        with h5py.File(io.BytesIO(b),'r') as f:
            v=f['cmorph']; raw=v[:,ii[0]:ii[-1]+1,jj[0]:jj[-1]+1].astype(np.float32); sf=float(v.attrs['scale_factor'][0]); mv=v.attrs['missing_value']
        raw[raw==float(np.atleast_1d(mv)[0])]=np.nan; raw[raw<0]=np.nan
        return (raw*sf).mean(0)
'''
open('prod/shard_dl.py','w').write(s.replace(old,new))
EOF
python - <<'EOF'
import sys; sys.argv=['x','cmorph','0','1']
src=open('prod/shard_dl.py').read(); src=src[:src.index('def work(k)')]
exec(src)
b=open('cm_test.nc','rb').read(); v=decode(b); import numpy as np
i=np.argmin(abs(la2[:,0]-25.8)); j=np.argmin(abs(lo2[0]+80.3)); print('test Miami',v[i,j])
EOF
for s in 0 1 2 3; do python prod/shard_dl.py cmorph $s 4 4 > prod/log_cm_$s.txt 2>&1 & done; wait; python prod/shard_dl.py cmorph -1 4 2>&1 | tail -2

test Miami 27.015
cmorph merged; missing chunks 0 hours with data 13420 of 13425


In [121]:
# [orig cell 120 | env: precip | status: ok]
NBc,mc_=load('cmorph'); print('cmorph lag',lagcorr_g(NBc))

cmorph lag {'metar': {-2: np.float64(0.192), -1: np.float64(0.291), 0: np.float64(0.384), 1: np.float64(0.288), 2: np.float64(0.191)}, 'onhour': {-2: np.float64(0.217), -1: np.float64(0.318), 0: np.float64(0.359), 1: np.float64(0.256), 2: np.float64(0.172)}}


In [122]:
%%bash
# [orig cell 121 | env: precip | status: ok]
for s in 0 1 2 3; do python prod/shard_dl.py pccs $s 4 4 > prod/log_pccs_$s.txt 2>&1 & done; wait; python prod/shard_dl.py pccs -1 4 2>&1 | tail -2

pccs merged; missing chunks 0 hours with data 13415 of 13425


In [124]:
# [orig cell 123 | env: precip | status: ok]
NBp,mp_=load('pccs'); print('pccs lag',lagcorr_g(NBp))
avail=[p for p in PRODS if os.path.exists(f'prod/{p}_win.npy')]
SCa,PEa,NBall=run(avail)
# common event set: event present (finite N1 lag0) for every product
PEa['key']=PEa.event_id
cnt=PEa[PEa.nbhd=='N1'].groupby(['threshold','key']).product.nunique()
common=set(cnt[cnt==len(avail)].index)
print({T:sum(1 for t,k in common if t==T) for T in EV})

pccs lag {'metar': {-2: np.float64(0.083), -1: np.float64(0.142), 0: np.float64(0.291), 1: np.float64(0.242), 2: np.float64(0.136)}, 'onhour': {-2: np.float64(0.093), -1: np.float64(0.17), 0: np.float64(0.295), 1: np.float64(0.206), 2: np.float64(0.123)}}
{20: 6950, 30: 2769, 40: 1055, 50: 446}
